# Modeling a Plant

## What is a Plant?

In control systems, a **plant** is the physical system being controlled — the thing whose behaviour we want to shape. Before we can control it, we need a mathematical model of how its state evolves over time.

In `dynamicalnodes`, every plant is represented as a [`DynamicalSystem(f=..., h=...)`](https://nehalsinghmangat.github.io/dynamicalnodes/api/dynamical_system.html) where:

- `f(x_k, ...) → x_{k+1}` — **state transition**: given the current state, compute the next state
- `h(x_k, ...) → y_k` — **observation**: given the current state, compute what a sensor would measure

This notebook models a **pendulum** — a standard nonlinear plant. Because the pendulum's equation of motion contains $\sin(\theta)$, it cannot be written directly as a linear discrete-time update. We must **linearize** first, then **discretize**. Two different ways of doing this are compared.

## Continuous-Time Pendulum

A pendulum of length $L$ and mass $m$ with viscous damping $c$ obeys the second-order ODE:

$$
\ddot{\theta} = -\frac{g}{L}\sin(\theta) - c\,\dot{\theta}
$$

Introducing state $x = [\theta,\; \dot{\theta}]^\top$, the continuous-time vector field is:

$$
\dot{x} = f_{\text{ct}}(x)
=
\begin{bmatrix} \dot{\theta} \\ -\dfrac{g}{L}\sin(\theta) - c\,\dot{\theta} \end{bmatrix}
$$

This is **nonlinear** because of $\sin(\theta)$. To build a `DynamicalSystem`, we need a *discrete-time* update $x_{k+1} = f(x_k)$. Getting there requires two steps: linearization, then discretization.

![Autonomous pendulum: pivot, rod of length L, mass m displaced by angle θ from vertical, with gravity g pointing downward.](../../_static/figures/pendulum_autonomous.svg)

## Step 1 — Linearization

Near the equilibrium $\theta = 0$ (hanging straight down), the small-angle approximation gives $\sin(\theta) \approx \theta$. This replaces the nonlinear ODE with a **linear** one:

$$
\dot{x} \approx A x,
\qquad
A = \begin{bmatrix} 0 & 1 \\ -g/L & -c \end{bmatrix}
$$

$A$ is the Jacobian $\partial f_{\text{ct}}/\partial x$ evaluated at $x = 0$. We now have a linear continuous-time system — but we still need to turn it into a discrete-time update.

## Step 2 — Discretization: Two Approaches

Given the linear CT model $\dot{x} = Ax$, we need a discrete-time matrix $F$ such that $x_{k+1} = F x_k$. Two common choices give different $F$:

### Method 1 — Forward Euler

Approximate the derivative as a finite difference:

$$\dot{x} \approx \frac{x_{k+1} - x_k}{\Delta t} \implies x_{k+1} = (I + \Delta t\, A)\, x_k$$

$$\boxed{F_{\text{Euler}} = I + \Delta t\, A}$$

This is a **first-order approximation** — it assumes the dynamics are constant over $[t_k,\, t_{k+1}]$. The error is $\mathcal{O}(\Delta t^2)$ per step.

### Method 2 — Zero-Order Hold (Matrix Exponential)

The *exact* solution to $\dot{x} = Ax$ with initial condition $x(t_k)$ is:

$$x(t_k + \Delta t) = e^{A \Delta t}\, x(t_k)$$

$$\boxed{F_{\text{ZOH}} = e^{A \Delta t}}$$

This is exact for the **linearized** model — no approximation in the time-stepping. $e^{A\Delta t}$ is computed via `scipy.linalg.expm`.

---

Both $F_{\text{Euler}}$ and $F_{\text{ZOH}}$ are linear maps, so both yield the same `DynamicalSystem` structure — only the matrix $F$ differs:

$$f(x_k, F) = F\, x_k$$

## Implementation

In [ ]:
import numpy as np
from scipy.linalg import expm
from scipy.integrate import solve_ivp
from dynamicalnodes import DynamicalSystem

### Parameters and Linearized System Matrix

In [ ]:
g = 9.81  # m/s²
L = 1.0   # m
c = 0.3   # damping coefficient (1/s)
dt = 0.15 # s  — deliberately coarse to make Euler vs ZOH differences visible

sim_time = np.arange(0, 8, dt)

# Linearized CT system matrix (Jacobian at θ=0)
A = np.array([[0.0,    1.0],
              [-g / L, -c]])

# Method 1: Euler discrete-time matrix
F_euler = np.eye(2) + dt * A

# Method 2: ZOH (exact) discrete-time matrix
F_zoh = expm(A * dt)

print("F_euler:\n", F_euler)
print("\nF_zoh:\n", F_zoh)

### DynamicalSystem Blocks

Both plants share the same `f` and `h` signatures — only the matrix `F` passed at runtime differs.

In [ ]:
def pendulum_f(xk, F):
    return F @ xk


def pendulum_h(xk):
    return xk[0]  # angle θ


euler_plant = DynamicalSystem(f=pendulum_f, h=pendulum_h)
zoh_plant   = DynamicalSystem(f=pendulum_f, h=pendulum_h)

### Reference Trajectory

To judge accuracy, we integrate the **full nonlinear ODE** with a high-resolution solver. This is our ground truth.

In [ ]:
theta0 = 0.4  # rad  (~23°  — small but not negligible)
x0 = np.array([theta0, 0.0])


def pendulum_ct(t, x):
    theta, thetadot = x
    return [thetadot, -(g / L) * np.sin(theta) - c * thetadot]


sol = solve_ivp(
    pendulum_ct,
    t_span=(sim_time[0], sim_time[-1]),
    y0=x0,
    t_eval=sim_time,
    method="RK45",
    rtol=1e-9,
)
theta_true = sol.y[0]

### Simulation

In [ ]:
xk_euler = x0.copy()
xk_zoh   = x0.copy()

theta_euler, theta_zoh = [], []

for _ in sim_time:
    xk_euler, th_e = euler_plant.step(xk=xk_euler, F=F_euler)
    xk_zoh,   th_z = zoh_plant.step(  xk=xk_zoh,   F=F_zoh)
    theta_euler.append(th_e)
    theta_zoh.append(th_z)

### Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

# ── Top: trajectory comparison ──────────────────────────────────────────────
axs[0].plot(sim_time, theta_true,  color="black",     linewidth=2,   label="Nonlinear ODE (RK45 reference)")
axs[0].plot(sim_time, theta_euler, color="tomato",    linewidth=1.5, linestyle="--", label="Euler  ($F = I + \\Delta t A$)")
axs[0].plot(sim_time, theta_zoh,   color="steelblue", linewidth=1.5, linestyle="-.", label="ZOH    ($F = e^{A\\Delta t}$)")
axs[0].set_ylabel("Angle $\\theta_k$ (rad)")
axs[0].set_title(f"Pendulum: linearized discrete-time models vs nonlinear reference  ($\\theta_0 = {theta0}$ rad, $\\Delta t = {dt}$ s)")
axs[0].legend()
axs[0].axhline(0, color="gray", linewidth=0.6, linestyle=":")

# ── Bottom: error vs reference ───────────────────────────────────────────────
axs[1].plot(sim_time, np.array(theta_euler) - theta_true, color="tomato",    linewidth=1.5, linestyle="--", label="Euler error")
axs[1].plot(sim_time, np.array(theta_zoh)   - theta_true, color="steelblue", linewidth=1.5, linestyle="-.", label="ZOH error")
axs[1].set_ylabel("Error vs reference (rad)")
axs[1].set_xlabel("Time (s)")
axs[1].set_title("Discretization error  (both methods linearized at $\\theta = 0$)")
axs[1].legend()
axs[1].axhline(0, color="gray", linewidth=0.6, linestyle=":")

plt.tight_layout()
plt.show()

## Reading the Plot

Both models diverge from the nonlinear reference for the same underlying reason — the **linearization** itself is approximate ($\sin\theta \approx \theta$ only holds near $\theta = 0$). But on top of that, **Euler accumulates additional discretization error** at each step that ZOH does not:

| | Source of error |
|---|---|
| ZOH | Linearization only (sin θ ≈ θ) |
| Euler | Linearization **+** $\mathcal{O}(\Delta t^2)$ per-step truncation |

The error panel makes this visible: the ZOH curve stays closer to the reference throughout. Shrinking $\Delta t$ would reduce Euler's truncation error and bring both curves together — at the cost of more timesteps.

The linearization error (shared by both) would only shrink if the initial angle $\theta_0$ were reduced or a higher-order Taylor expansion were used.

## Summary

Modeling a plant with `DynamicalSystem` requires a discrete-time `f`. For nonlinear plants, that means two choices:

1. **Linearize** (replace $\sin\theta$ with $\theta$) to obtain the system matrix $A$.
2. **Discretize** $A$ into $F$ — either approximately (Euler: $F = I + \Delta t A$) or exactly for the linear model (ZOH: $F = e^{A\Delta t}$).

Both end up as the same one-line `DynamicalSystem`:

```python
DynamicalSystem(f=lambda xk, F: F @ xk, h=lambda xk: xk[0])
```

The choice of $F$ lives outside the `DynamicalSystem` — in how you compute the transition matrix before simulation.

The next notebook, [Modeling a Plant with Input](plant_with_input.ipynb), adds an external control input $u_k$ to the pendulum.